In [5]:
import os
import json
import time
import shutil
import re
from pathlib import Path
from collections import Counter

import pandas as pd

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [6]:
PROJECT_ROOT = Path(r"C:\Users\asguug\Documents\rag-agent")

if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DOCS_DIR = PROJECT_ROOT / "data" / "docs"
EVAL_DIR = PROJECT_ROOT / "data" / "eval"
CHROMA_WEEK5_DIR = PROJECT_ROOT / "data" / "chroma_week5"

EVAL_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_WEEK5_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DOCS_DIR:", DOCS_DIR)
print("EVAL_DIR:", EVAL_DIR)
print("CHROMA_WEEK5_DIR:", CHROMA_WEEK5_DIR)
print("OPENAI_API_KEY exists:", bool(os.getenv("OPENAI_API_KEY")))

PROJECT_ROOT: C:\Users\asguug\Documents\rag-agent
DOCS_DIR: C:\Users\asguug\Documents\rag-agent\data\docs
EVAL_DIR: C:\Users\asguug\Documents\rag-agent\data\eval
CHROMA_WEEK5_DIR: C:\Users\asguug\Documents\rag-agent\data\chroma_week5
OPENAI_API_KEY exists: True


In [7]:
md_files = sorted(DOCS_DIR.glob("*.md"))

print("Markdown 문서 수:", len(md_files))

for path in md_files:
    print("-", path.name)

assert md_files, f"Markdown 문서가 없습니다. 경로 확인 필요: {DOCS_DIR}"


raw_docs = []

for path in md_files:
    raw_docs.append(
        Document(
            page_content=path.read_text(encoding="utf-8"),
            metadata={
                "source": str(path),
                "source_file": path.name,
                "game_key": path.stem,
                "document_type": "game_profile",
                "source_type": "steam",
            },
        )
    )

print("로드된 문서 수:", len(raw_docs))

Markdown 문서 수: 5
- baldurs_gate_3.md
- cyberpunk_2077.md
- hollow_knight.md
- monster_hunter_world.md
- no_mans_sky.md
로드된 문서 수: 5


In [8]:
def split_markdown_by_fixed_sections(doc: Document) -> list[Document]:
    """
    Steam 게임 Markdown 문서를 고정 section 단위로 분리한다.

    사용 section:
    - metadata
    - store_summary
    - about
    - review
    - news
    """
    source = doc.metadata.get("source", "")
    source_file = doc.metadata.get("source_file", Path(source).name)
    game_key = doc.metadata.get("game_key", Path(source_file).stem)

    text = doc.page_content

    section_patterns = [
        ("metadata", "## Metadata"),
        ("store_summary", "## Store Summary"),
        ("about", "## About The Game"),
        ("review", "## Recent Steam Reviews"),
        ("news", "## Steam News and Updates"),
    ]

    section_docs = []

    for idx, (section_name, heading) in enumerate(section_patterns):
        start = text.find(heading)

        if start == -1:
            continue

        if idx + 1 < len(section_patterns):
            next_heading = section_patterns[idx + 1][1]
            end = text.find(next_heading, start + len(heading))

            if end == -1:
                end = len(text)
        else:
            end = len(text)

        section_text = text[start:end].strip()

        if not section_text:
            continue

        section_docs.append(
            Document(
                page_content=section_text,
                metadata={
                    **doc.metadata,
                    "source": source,
                    "source_file": source_file,
                    "game_key": game_key,
                    "section": section_name,
                    "document_type": "game_profile",
                    "source_type": "steam",
                },
            )
        )

    return section_docs


section_docs = []

for doc in raw_docs:
    section_docs.extend(split_markdown_by_fixed_sections(doc))

print("원본 Markdown 문서 수:", len(raw_docs))
print("섹션 문서 수:", len(section_docs))

section_counter = Counter(doc.metadata.get("section", "unknown") for doc in section_docs)
print(section_counter)

for doc in section_docs[:5]:
    print(doc.metadata["source_file"], doc.metadata["section"], len(doc.page_content))

원본 Markdown 문서 수: 5
섹션 문서 수: 25
Counter({'metadata': 5, 'store_summary': 5, 'about': 5, 'review': 5, 'news': 5})
baldurs_gate_3.md metadata 636
baldurs_gate_3.md store_summary 224
baldurs_gate_3.md about 5376
baldurs_gate_3.md review 5101
baldurs_gate_3.md news 2704


In [9]:
def normalize_section_name(section_title: str) -> str:
    section_title = str(section_title).strip().lower()

    if section_title in ["metadata", "store_summary", "about", "review", "news"]:
        return section_title

    if "metadata" in section_title:
        return "metadata"

    if "store summary" in section_title:
        return "store_summary"

    if "about" in section_title:
        return "about"

    if "review" in section_title:
        return "review"

    if "news" in section_title or "update" in section_title:
        return "news"

    return "unknown"


def add_chunk_metadata(
    chunks: list[Document],
    strategy_name: str,
    chunk_size,
    chunk_overlap,
) -> list[Document]:
    """
    chunk별 공통 metadata를 보강한다.
    """
    enriched = []

    for idx, chunk in enumerate(chunks):
        metadata = dict(chunk.metadata)

        metadata["chunk_strategy"] = strategy_name
        metadata["chunk_index"] = idx
        metadata["chunk_size_setting"] = str(chunk_size)
        metadata["chunk_overlap_setting"] = str(chunk_overlap)
        metadata["char_count"] = len(chunk.page_content)

        if "source_file" not in metadata:
            metadata["source_file"] = Path(metadata.get("source", "")).name

        if "game_key" not in metadata:
            metadata["game_key"] = Path(metadata.get("source_file", "")).stem

        metadata["section"] = normalize_section_name(metadata.get("section", "unknown"))

        if "document_type" not in metadata:
            metadata["document_type"] = "game_profile"

        if "source_type" not in metadata:
            metadata["source_type"] = "steam"

        enriched.append(
            Document(
                page_content=chunk.page_content,
                metadata=metadata,
            )
        )

    return enriched

In [10]:
WEEK5_BASELINE_STRATEGY = "B_recursive_large_1000_200"


def make_chunks_strategy_b(section_docs: list[Document]) -> list[Document]:
    """
    4주차 최종 선택 전략 B:
    section 분리 후 RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )

    chunks = splitter.split_documents(section_docs)

    return add_chunk_metadata(
        chunks=chunks,
        strategy_name=WEEK5_BASELINE_STRATEGY,
        chunk_size=1000,
        chunk_overlap=200,
    )


chunks_b = make_chunks_strategy_b(section_docs)

print("전략 B chunk 수:", len(chunks_b))
print("첫 번째 chunk metadata:")
print(chunks_b[0].metadata)
print("\n첫 번째 chunk preview:")
print(chunks_b[0].page_content[:500])

전략 B chunk 수: 95
첫 번째 chunk metadata:
{'source': 'C:\\Users\\asguug\\Documents\\rag-agent\\data\\docs\\baldurs_gate_3.md', 'source_file': 'baldurs_gate_3.md', 'game_key': 'baldurs_gate_3', 'document_type': 'game_profile', 'source_type': 'steam', 'section': 'metadata', 'chunk_strategy': 'B_recursive_large_1000_200', 'chunk_index': 0, 'chunk_size_setting': '1000', 'chunk_overlap_setting': '200', 'char_count': 636}

첫 번째 chunk preview:
## Metadata
- game_key: baldurs_gate_3
- appid: 1086940
- title: Baldur's Gate 3
- release_date: Aug 3, 2023
- developers: Larian Studios
- publishers: Larian Studios
- genres: Adventure, RPG, Strategy
- categories: Single-player, Multi-player, Co-op, Online Co-op, LAN Co-op, Cross-Platform Multiplayer, Steam Achievements, Full controller support, Steam Trading Cards, Adjustable Text Size, Camera Comfort, Color Alternatives, Custom Volume Controls, Adjustable Difficulty, Playable without Timed I


In [11]:
chunk_stat_records = []

lengths = [len(chunk.page_content) for chunk in chunks_b]
section_counter = Counter(chunk.metadata.get("section", "unknown") for chunk in chunks_b)
game_counter = Counter(chunk.metadata.get("game_key", "unknown") for chunk in chunks_b)

chunk_stat_records.append(
    {
        "strategy": WEEK5_BASELINE_STRATEGY,
        "chunk_count": len(chunks_b),
        "min_chars": min(lengths),
        "max_chars": max(lengths),
        "avg_chars": round(sum(lengths) / len(lengths), 1),
        "section_distribution": json.dumps(dict(section_counter), ensure_ascii=False),
        "game_distribution": json.dumps(dict(game_counter), ensure_ascii=False),
    }
)

chunk_stats_df = pd.DataFrame(chunk_stat_records)

display(chunk_stats_df)

chunk_stats_path = EVAL_DIR / "week5_baseline_chunk_stats.csv"
chunk_stats_df.to_csv(chunk_stats_path, index=False, encoding="utf-8-sig")

print("saved:", chunk_stats_path)

,strategy,chunk_count,min_chars,max_chars,avg_chars,section_distribution,game_distribution
0,B_recursive_large_1000_200,95,62,999,707.5,"{""metadata"": 5, ""store_summary"": 5, ""about"": 2...","{""baldurs_gate_3"": 19, ""cyberpunk_2077"": 11, ""..."


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_baseline_chunk_stats.csv


In [12]:
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [13]:
DENSE_BASELINE_COLLECTION = "week5_dense_baseline_b_1000_200"
DENSE_BASELINE_DIR = CHROMA_WEEK5_DIR / DENSE_BASELINE_COLLECTION

if DENSE_BASELINE_DIR.exists():
    shutil.rmtree(DENSE_BASELINE_DIR)

DENSE_BASELINE_DIR.mkdir(parents=True, exist_ok=True)

dense_vectorstore = Chroma.from_documents(
    documents=chunks_b,
    embedding=embeddings,
    collection_name=DENSE_BASELINE_COLLECTION,
    persist_directory=str(DENSE_BASELINE_DIR),
)

print("Dense baseline vectorstore created")
print("persist_directory:", DENSE_BASELINE_DIR)
print("chunk_count:", len(chunks_b))

Dense baseline vectorstore created
persist_directory: C:\Users\asguug\Documents\rag-agent\data\chroma_week5\week5_dense_baseline_b_1000_200
chunk_count: 95


In [14]:
def preview_docs(docs, max_chars: int = 250):
    rows = []

    for rank, doc in enumerate(docs, start=1):
        rows.append(
            {
                "rank": rank,
                "source_file": doc.metadata.get("source_file"),
                "game_key": doc.metadata.get("game_key"),
                "section": doc.metadata.get("section"),
                "chunk_index": doc.metadata.get("chunk_index"),
                "preview": doc.page_content[:max_chars].replace("\n", " "),
            }
        )

    return pd.DataFrame(rows)


test_question = "Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?"

dense_docs = dense_vectorstore.similarity_search(
    query=test_question,
    k=5,
)

display(preview_docs(dense_docs))

,rank,source_file,game_key,section,chunk_index,preview
0,1,monster_hunter_world.md,monster_hunter_world,news,59,{STEAM_CLAN_IMAGE}/45725708/73392ac8e0f10e408b...
1,2,monster_hunter_world.md,monster_hunter_world,store_summary,47,## Store Summary Welcome to a new world! In Mo...
2,3,monster_hunter_world.md,monster_hunter_world,news,57,## Steam News and Updates ### News 1: Monster ...
3,4,monster_hunter_world.md,monster_hunter_world,about,48,## About The Game Welcome to a new world! Take...
4,5,monster_hunter_world.md,monster_hunter_world,news,64,### News 4: Pre-Order Monster Hunter Stories 3...


In [15]:
eval_questions = [
    "Hollow Knight는 어떤 플레이 스타일의 게임인가요?",
    "Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?",
    "Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?",
    "Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?",
    "Baldur's Gate 3는 어떤 RPG인가요?",
    "Baldur's Gate 3에서 선택과 서사는 어떤 역할을 하나요?",
    "No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?",
    "No Man's Sky의 최근 뉴스나 업데이트 방향은 무엇인가요?",
    "Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?",
    "Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?",
]

pd.DataFrame({"question": eval_questions})

,question
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?
1,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?
2,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?
3,Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?
4,Baldur's Gate 3는 어떤 RPG인가요?
5,Baldur's Gate 3에서 선택과 서사는 어떤 역할을 하나요?
6,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?
7,No Man's Sky의 최근 뉴스나 업데이트 방향은 무엇인가요?
8,Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?
9,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?


In [16]:
def infer_intent(question: str) -> str:
    """
    질문의 retrieval intent를 rule 기반으로 단순 분류한다.

    반환값:
    - gameplay: 플레이 스타일, 전투, 분위기, 핵심 루프 등
    - review: 리뷰, 평가, 반응, 민심 등
    - news: 업데이트, 패치, 뉴스 등
    - general: 그 외 일반 질문
    """
    q = question.lower()

    review_keywords = [
        "리뷰", "평가", "반응", "민심", "여론",
        "review", "reviews", "sentiment",
    ]

    news_keywords = [
        "업데이트", "패치", "뉴스", "최근 뉴스",
        "update", "updates", "patch", "news",
    ]

    gameplay_keywords = [
        "플레이", "전투", "루프", "스타일", "분위기", "월드",
        "협동", "선택", "서사", "특징", "rpg",
        "gameplay", "combat", "style", "world", "co-op",
    ]

    if any(keyword in q for keyword in review_keywords):
        return "review"

    if any(keyword in q for keyword in news_keywords):
        return "news"

    if any(keyword in q for keyword in gameplay_keywords):
        return "gameplay"

    return "general"


intent_rows = []

for question in eval_questions:
    intent_rows.append(
        {
            "question": question,
            "intent": infer_intent(question),
        }
    )

pd.DataFrame(intent_rows)

,question,intent
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay
1,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,gameplay
2,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,gameplay
3,Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?,gameplay
4,Baldur's Gate 3는 어떤 RPG인가요?,gameplay
5,Baldur's Gate 3에서 선택과 서사는 어떤 역할을 하나요?,gameplay
6,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,news
7,No Man's Sky의 최근 뉴스나 업데이트 방향은 무엇인가요?,news
8,Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?,review
9,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,gameplay


In [17]:
GAME_ALIASES = {
    "hollow knight": "hollow_knight",
    "할로우 나이트": "hollow_knight",

    "monster hunter: world": "monster_hunter_world",
    "monster hunter world": "monster_hunter_world",
    "monster hunter": "monster_hunter_world",
    "몬스터 헌터": "monster_hunter_world",

    "baldur's gate 3": "baldurs_gate_3",
    "baldurs gate 3": "baldurs_gate_3",
    "baldur": "baldurs_gate_3",
    "발더스": "baldurs_gate_3",

    "no man's sky": "no_mans_sky",
    "no mans sky": "no_mans_sky",
    "노 맨즈 스카이": "no_mans_sky",

    "cyberpunk 2077": "cyberpunk_2077",
    "cyberpunk": "cyberpunk_2077",
    "사이버펑크": "cyberpunk_2077",
}


def infer_game_key(question: str):
    q = question.lower()

    for alias, game_key in GAME_ALIASES.items():
        if alias in q:
            return game_key

    return None


game_rows = []

for question in eval_questions:
    game_rows.append(
        {
            "question": question,
            "game_key": infer_game_key(question),
        }
    )

pd.DataFrame(game_rows)

,question,game_key
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,hollow_knight
1,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,hollow_knight
2,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,monster_hunter_world
3,Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?,monster_hunter_world
4,Baldur's Gate 3는 어떤 RPG인가요?,baldurs_gate_3
5,Baldur's Gate 3에서 선택과 서사는 어떤 역할을 하나요?,baldurs_gate_3
6,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,no_mans_sky
7,No Man's Sky의 최근 뉴스나 업데이트 방향은 무엇인가요?,no_mans_sky
8,Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?,cyberpunk_2077
9,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,cyberpunk_2077


In [18]:
def build_metadata_filter(question: str):
    """
    질문에서 game_key와 intent를 추정한 뒤 Chroma metadata filter를 만든다.
    """
    intent = infer_intent(question)
    game_key = infer_game_key(question)

    conditions = []

    if game_key:
        conditions.append({"game_key": {"$eq": game_key}})

    if intent == "review":
        conditions.append({"section": {"$eq": "review"}})

    elif intent == "news":
        conditions.append({"section": {"$eq": "news"}})

    elif intent == "gameplay":
        conditions.append(
            {
                "$or": [
                    {"section": {"$eq": "about"}},
                    {"section": {"$eq": "store_summary"}},
                    {"section": {"$eq": "metadata"}},
                ]
            }
        )

    if not conditions:
        return None

    if len(conditions) == 1:
        return conditions[0]

    return {"$and": conditions}


filter_rows = []

for question in eval_questions:
    filter_rows.append(
        {
            "question": question,
            "intent": infer_intent(question),
            "game_key": infer_game_key(question),
            "filter": build_metadata_filter(question),
        }
    )

pd.DataFrame(filter_rows)

,question,intent,game_key,filter
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,hollow_knight,{'$and': [{'game_key': {'$eq': 'hollow_knight'...
1,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,gameplay,hollow_knight,{'$and': [{'game_key': {'$eq': 'hollow_knight'...
2,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,gameplay,monster_hunter_world,{'$and': [{'game_key': {'$eq': 'monster_hunter...
3,Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?,gameplay,monster_hunter_world,{'$and': [{'game_key': {'$eq': 'monster_hunter...
4,Baldur's Gate 3는 어떤 RPG인가요?,gameplay,baldurs_gate_3,{'$and': [{'game_key': {'$eq': 'baldurs_gate_3...
5,Baldur's Gate 3에서 선택과 서사는 어떤 역할을 하나요?,gameplay,baldurs_gate_3,{'$and': [{'game_key': {'$eq': 'baldurs_gate_3...
6,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,news,no_mans_sky,{'$and': [{'game_key': {'$eq': 'no_mans_sky'}}...
7,No Man's Sky의 최근 뉴스나 업데이트 방향은 무엇인가요?,news,no_mans_sky,{'$and': [{'game_key': {'$eq': 'no_mans_sky'}}...
8,Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?,review,cyberpunk_2077,{'$and': [{'game_key': {'$eq': 'cyberpunk_2077...
9,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,gameplay,cyberpunk_2077,{'$and': [{'game_key': {'$eq': 'cyberpunk_2077...


In [19]:
def dense_retrieve(question: str, k: int = 5):
    """
    순수 dense similarity search.
    5주차 ablation의 A(Dense only)에 해당한다.
    """
    return dense_vectorstore.similarity_search(
        query=question,
        k=k,
    )


def dense_filter_retrieve(question: str, k: int = 5):
    """
    Metadata Filtering을 적용한 dense search.
    필수 ablation 4종과 별도로 보조 실험에 사용한다.
    """
    metadata_filter = build_metadata_filter(question)

    if metadata_filter:
        return dense_vectorstore.similarity_search(
            query=question,
            k=k,
            filter=metadata_filter,
        )

    return dense_vectorstore.similarity_search(
        query=question,
        k=k,
    )

In [20]:
def compare_retrieval(question: str, k: int = 5):
    dense_docs = dense_retrieve(question, k=k)
    filtered_docs = dense_filter_retrieve(question, k=k)

    dense_df = preview_docs(dense_docs)
    dense_df.insert(0, "retriever", "dense_only")

    filtered_df = preview_docs(filtered_docs)
    filtered_df.insert(0, "retriever", "dense_metadata_filter")

    return pd.concat([dense_df, filtered_df], ignore_index=True)


test_question = "Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?"

comparison_df = compare_retrieval(test_question, k=5)
display(comparison_df)

,retriever,rank,source_file,game_key,section,chunk_index,preview
0,dense_only,1,monster_hunter_world.md,monster_hunter_world,news,59,{STEAM_CLAN_IMAGE}/45725708/73392ac8e0f10e408b...
1,dense_only,2,monster_hunter_world.md,monster_hunter_world,store_summary,47,## Store Summary Welcome to a new world! In Mo...
2,dense_only,3,monster_hunter_world.md,monster_hunter_world,news,57,## Steam News and Updates ### News 1: Monster ...
3,dense_only,4,monster_hunter_world.md,monster_hunter_world,about,48,## About The Game Welcome to a new world! Take...
4,dense_only,5,monster_hunter_world.md,monster_hunter_world,news,64,### News 4: Pre-Order Monster Hunter Stories 3...
5,dense_metadata_filter,1,monster_hunter_world.md,monster_hunter_world,store_summary,47,## Store Summary Welcome to a new world! In Mo...
6,dense_metadata_filter,2,monster_hunter_world.md,monster_hunter_world,about,48,## About The Game Welcome to a new world! Take...
7,dense_metadata_filter,3,monster_hunter_world.md,monster_hunter_world,metadata,46,## Metadata - game_key: monster_hunter_world -...
8,dense_metadata_filter,4,monster_hunter_world.md,monster_hunter_world,about,50,"HUNTING A Diverse Arsenal, and an Indispensabl..."
9,dense_metadata_filter,5,monster_hunter_world.md,monster_hunter_world,about,51,"From diversion tactics to creating shortcuts, ..."


In [21]:
filter_eval_rows = []

for question in eval_questions:
    docs = dense_filter_retrieve(question, k=5)

    for rank, doc in enumerate(docs, start=1):
        filter_eval_rows.append(
            {
                "question": question,
                "intent": infer_intent(question),
                "inferred_game_key": infer_game_key(question),
                "rank": rank,
                "source_file": doc.metadata.get("source_file"),
                "game_key": doc.metadata.get("game_key"),
                "section": doc.metadata.get("section"),
                "chunk_index": doc.metadata.get("chunk_index"),
                "preview": doc.page_content[:300].replace("\n", " "),
            }
        )

filter_eval_df = pd.DataFrame(filter_eval_rows)

display(filter_eval_df)

filter_eval_path = EVAL_DIR / "week5_dense_metadata_filter_results.csv"
filter_eval_df.to_csv(filter_eval_path, index=False, encoding="utf-8-sig")

print("saved:", filter_eval_path)

,question,intent,inferred_game_key,rank,source_file,game_key,section,chunk_index,preview
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,hollow_knight,1,hollow_knight.md,hollow_knight,store_summary,31,## Store Summary Forge your own path in Hollow...
1,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,hollow_knight,2,hollow_knight.md,hollow_knight,about,35,Complete Hollow Knight to unlock Steel Soul Mo...
2,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,hollow_knight,3,hollow_knight.md,hollow_knight,metadata,30,## Metadata - game_key: hollow_knight - appid:...
3,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,hollow_knight,4,hollow_knight.md,hollow_knight,about,32,## About The Game Hollow Knight Expands with F...
4,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,hollow_knight,5,hollow_knight.md,hollow_knight,about,33,"Game Features Classic side-scrolling action, w..."
5,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,gameplay,hollow_knight,1,hollow_knight.md,hollow_knight,about,35,Complete Hollow Knight to unlock Steel Soul Mo...
6,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,gameplay,hollow_knight,2,hollow_knight.md,hollow_knight,store_summary,31,## Store Summary Forge your own path in Hollow...
7,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,gameplay,hollow_knight,3,hollow_knight.md,hollow_knight,about,32,## About The Game Hollow Knight Expands with F...
8,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,gameplay,hollow_knight,4,hollow_knight.md,hollow_knight,metadata,30,## Metadata - game_key: hollow_knight - appid:...
9,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,gameplay,hollow_knight,5,hollow_knight.md,hollow_knight,about,34,An enormous cast of cute and creepy characters...


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_dense_metadata_filter_results.csv


In [22]:
section_compare_rows = []

for question in eval_questions:
    dense_docs = dense_retrieve(question, k=5)
    filtered_docs = dense_filter_retrieve(question, k=5)

    dense_sections = [doc.metadata.get("section", "unknown") for doc in dense_docs]
    filtered_sections = [doc.metadata.get("section", "unknown") for doc in filtered_docs]

    section_compare_rows.append(
        {
            "question": question,
            "intent": infer_intent(question),
            "dense_sections": dense_sections,
            "filtered_sections": filtered_sections,
        }
    )

section_compare_df = pd.DataFrame(section_compare_rows)
display(section_compare_df)

section_compare_path = EVAL_DIR / "week5_section_filter_comparison.csv"
section_compare_df.to_csv(section_compare_path, index=False, encoding="utf-8-sig")

print("saved:", section_compare_path)

,question,intent,dense_sections,filtered_sections
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,"[store_summary, about, metadata, news, about]","[store_summary, about, metadata, about, about]"
1,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,gameplay,"[about, store_summary, about, news, news]","[about, store_summary, about, metadata, about]"
2,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,gameplay,"[news, store_summary, news, about, news]","[store_summary, about, metadata, about, about]"
3,Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?,gameplay,"[about, store_summary, news, about, news]","[about, store_summary, metadata, about, about]"
4,Baldur's Gate 3는 어떤 RPG인가요?,gameplay,"[store_summary, review, metadata, about, news]","[store_summary, metadata, about, about, about]"
5,Baldur's Gate 3에서 선택과 서사는 어떤 역할을 하나요?,gameplay,"[store_summary, news, about, metadata, news]","[store_summary, about, metadata, about, about]"
6,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,news,"[news, news, news, news, store_summary]","[news, news, news, news, news]"
7,No Man's Sky의 최근 뉴스나 업데이트 방향은 무엇인가요?,news,"[news, news, news, news, news]","[news, news, news, news, news]"
8,Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?,review,"[news, about, store_summary, review, metadata]","[review, review, review, review, review]"
9,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,gameplay,"[about, store_summary, news, metadata, review]","[about, store_summary, metadata]"


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_section_filter_comparison.csv


In [23]:
try:
    from langchain_experimental.text_splitter import SemanticChunker

    semantic_splitter = SemanticChunker(
        embeddings=embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=95,
    )

    semantic_chunks = semantic_splitter.split_documents(section_docs)

    semantic_chunks = add_chunk_metadata(
        chunks=semantic_chunks,
        strategy_name="semantic_chunker_percentile_95",
        chunk_size="semantic",
        chunk_overlap="semantic",
    )

    print("SemanticChunker chunk 수:", len(semantic_chunks))
    print("첫 번째 semantic chunk metadata:")
    print(semantic_chunks[0].metadata)
    print("\n첫 번째 semantic chunk preview:")
    print(semantic_chunks[0].page_content[:500])

except Exception as e:
    semantic_chunks = []
    print("SemanticChunker 실행 실패")
    print(type(e).__name__, e)

C:\Users\asguug\AppData\Local\Temp\ipykernel_16860\2398972792.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


SemanticChunker chunk 수: 54
첫 번째 semantic chunk metadata:
{'source': 'C:\\Users\\asguug\\Documents\\rag-agent\\data\\docs\\baldurs_gate_3.md', 'source_file': 'baldurs_gate_3.md', 'game_key': 'baldurs_gate_3', 'document_type': 'game_profile', 'source_type': 'steam', 'section': 'metadata', 'chunk_strategy': 'semantic_chunker_percentile_95', 'chunk_index': 0, 'chunk_size_setting': 'semantic', 'chunk_overlap_setting': 'semantic', 'char_count': 636}

첫 번째 semantic chunk preview:
## Metadata
- game_key: baldurs_gate_3
- appid: 1086940
- title: Baldur's Gate 3
- release_date: Aug 3, 2023
- developers: Larian Studios
- publishers: Larian Studios
- genres: Adventure, RPG, Strategy
- categories: Single-player, Multi-player, Co-op, Online Co-op, LAN Co-op, Cross-Platform Multiplayer, Steam Achievements, Full controller support, Steam Trading Cards, Adjustable Text Size, Camera Comfort, Color Alternatives, Custom Volume Controls, Adjustable Difficulty, Playable without Timed I


In [24]:
if semantic_chunks:
    semantic_lengths = [len(chunk.page_content) for chunk in semantic_chunks]
    semantic_section_counter = Counter(chunk.metadata.get("section", "unknown") for chunk in semantic_chunks)
    semantic_game_counter = Counter(chunk.metadata.get("game_key", "unknown") for chunk in semantic_chunks)

    semantic_stats_df = pd.DataFrame(
        [
            {
                "strategy": "semantic_chunker_percentile_95",
                "chunk_count": len(semantic_chunks),
                "min_chars": min(semantic_lengths),
                "max_chars": max(semantic_lengths),
                "avg_chars": round(sum(semantic_lengths) / len(semantic_lengths), 1),
                "section_distribution": json.dumps(dict(semantic_section_counter), ensure_ascii=False),
                "game_distribution": json.dumps(dict(semantic_game_counter), ensure_ascii=False),
            }
        ]
    )

    display(semantic_stats_df)

    semantic_stats_path = EVAL_DIR / "week5_semantic_chunker_stats.csv"
    semantic_stats_df.to_csv(semantic_stats_path, index=False, encoding="utf-8-sig")

    print("saved:", semantic_stats_path)

else:
    print("SemanticChunker 결과가 없어 통계 저장을 건너뜁니다.")

,strategy,chunk_count,min_chars,max_chars,avg_chars,section_distribution,game_distribution
0,semantic_chunker_percentile_95,54,14,4860,1136.7,"{""metadata"": 6, ""store_summary"": 6, ""about"": 1...","{""baldurs_gate_3"": 12, ""cyberpunk_2077"": 7, ""h..."


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_semantic_chunker_stats.csv


In [25]:
compare_chunk_stats = [chunk_stats_df]

if semantic_chunks:
    compare_chunk_stats.append(semantic_stats_df)

compare_chunk_stats_df = pd.concat(compare_chunk_stats, ignore_index=True)

display(compare_chunk_stats_df)

compare_chunk_stats_path = EVAL_DIR / "week5_chunk_strategy_aux_comparison.csv"
compare_chunk_stats_df.to_csv(compare_chunk_stats_path, index=False, encoding="utf-8-sig")

print("saved:", compare_chunk_stats_path)

,strategy,chunk_count,min_chars,max_chars,avg_chars,section_distribution,game_distribution
0,B_recursive_large_1000_200,95,62,999,707.5,"{""metadata"": 5, ""store_summary"": 5, ""about"": 2...","{""baldurs_gate_3"": 19, ""cyberpunk_2077"": 11, ""..."
1,semantic_chunker_percentile_95,54,14,4860,1136.7,"{""metadata"": 6, ""store_summary"": 6, ""about"": 1...","{""baldurs_gate_3"": 12, ""cyberpunk_2077"": 7, ""h..."


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_chunk_strategy_aux_comparison.csv


In [26]:
from rank_bm25 import BM25Okapi
import numpy as np

def simple_tokenize(text: str) -> list[str]:
    """
    BM25용 단순 tokenizer.

    - 영어/숫자/한글 토큰 추출
    - 소문자 변환
    - Steam 게임명, 패치명, 태그명 같은 keyword matching을 우선 고려
    """
    text = str(text).lower()
    tokens = re.findall(r"[a-zA-Z0-9가-힣]+", text)
    return tokens


def expand_query_for_bm25(question: str) -> str:
    """
    한국어 질문이 영어 문서와 조금이라도 더 잘 맞도록
    intent별 영어 힌트 키워드를 추가한다.

    Dense retrieval은 의미 검색을 담당하고,
    BM25는 게임명/패치명/태그명 같은 정확한 키워드 매칭을 보완한다.
    """
    intent = infer_intent(question)

    expansions = {
        "gameplay": " gameplay combat loop play style feature world atmosphere co-op story choice rpg",
        "review": " review reviews recent positive negative player sentiment user feedback",
        "news": " update patch news announcement hotfix version release",
        "general": " game steam store description feature",
    }

    return question + " " + expansions.get(intent, expansions["general"])

In [27]:
bm25_corpus = [doc.page_content for doc in chunks_b]
tokenized_corpus = [simple_tokenize(text) for text in bm25_corpus]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 corpus size:", len(bm25_corpus))
print("Example tokenized doc length:", len(tokenized_corpus[0]))
print("Example tokens:", tokenized_corpus[0][:30])

BM25 corpus size: 95
Example tokenized doc length: 87
Example tokens: ['metadata', 'game', 'key', 'baldurs', 'gate', '3', 'appid', '1086940', 'title', 'baldur', 's', 'gate', '3', 'release', 'date', 'aug', '3', '2023', 'developers', 'larian', 'studios', 'publishers', 'larian', 'studios', 'genres', 'adventure', 'rpg', 'strategy', 'categories', 'single']


In [28]:
def get_doc_id(doc: Document) -> str:
    """
    Dense / BM25 결과를 합칠 때 사용할 chunk 고유 ID.
    """
    metadata = doc.metadata

    return "::".join(
        [
            str(metadata.get("source_file", "")),
            str(metadata.get("game_key", "")),
            str(metadata.get("section", "")),
            str(metadata.get("chunk_index", "")),
        ]
    )

In [29]:
def retrieve_dense_only(question: str, k: int = 5) -> list[Document]:
    """
    A. Dense only

    4주차 최종 chunking 전략 B로 만든 Chroma vector store에서
    순수 similarity search만 수행한다.
    """
    return dense_vectorstore.similarity_search(
        query=question,
        k=k,
    )

In [30]:
def retrieve_bm25_only(question: str, k: int = 5) -> list[Document]:
    """
    B. BM25 only

    rank_bm25를 사용해 keyword 기반 검색을 수행한다.
    """
    expanded_question = expand_query_for_bm25(question)
    query_tokens = simple_tokenize(expanded_question)

    scores = bm25.get_scores(query_tokens)

    ranked_indices = np.argsort(scores)[::-1][:k]

    return [chunks_b[int(i)] for i in ranked_indices]

In [31]:
test_question = "Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?"

dense_docs = retrieve_dense_only(test_question, k=5)
bm25_docs = retrieve_bm25_only(test_question, k=5)

dense_df = preview_docs(dense_docs)
dense_df.insert(0, "retriever", "dense_only")

bm25_df = preview_docs(bm25_docs)
bm25_df.insert(0, "retriever", "bm25_only")

display(pd.concat([dense_df, bm25_df], ignore_index=True))

,retriever,rank,source_file,game_key,section,chunk_index,preview
0,dense_only,1,monster_hunter_world.md,monster_hunter_world,news,59,{STEAM_CLAN_IMAGE}/45725708/73392ac8e0f10e408b...
1,dense_only,2,monster_hunter_world.md,monster_hunter_world,store_summary,47,## Store Summary Welcome to a new world! In Mo...
2,dense_only,3,monster_hunter_world.md,monster_hunter_world,news,57,## Steam News and Updates ### News 1: Monster ...
3,dense_only,4,monster_hunter_world.md,monster_hunter_world,about,48,## About The Game Welcome to a new world! Take...
4,dense_only,5,monster_hunter_world.md,monster_hunter_world,news,64,### News 4: Pre-Order Monster Hunter Stories 3...
5,bm25_only,1,monster_hunter_world.md,monster_hunter_world,metadata,46,## Metadata - game_key: monster_hunter_world -...
6,bm25_only,2,baldurs_gate_3.md,baldurs_gate_3,metadata,0,## Metadata - game_key: baldurs_gate_3 - appid...
7,bm25_only,3,no_mans_sky.md,no_mans_sky,metadata,70,## Metadata - game_key: no_mans_sky - appid: 2...
8,bm25_only,4,monster_hunter_world.md,monster_hunter_world,about,48,## About The Game Welcome to a new world! Take...
9,bm25_only,5,baldurs_gate_3.md,baldurs_gate_3,about,4,"The Forgotten Realms are a vast, detailed, and..."


In [32]:
def reciprocal_rank_fusion(
    ranked_doc_lists: list[list[Document]],
    weights: list[float] | None = None,
    rrf_k: int = 60,
    top_n: int = 5,
) -> list[Document]:
    """
    여러 retriever의 ranking 결과를 RRF로 결합한다.

    Args:
        ranked_doc_lists:
            retriever별 Document ranking 리스트
        weights:
            retriever별 가중치. None이면 동일 가중치.
        rrf_k:
            RRF rank constant. 일반적으로 60을 많이 사용.
        top_n:
            최종 반환할 문서 수
    """
    if weights is None:
        weights = [1.0] * len(ranked_doc_lists)

    assert len(ranked_doc_lists) == len(weights), "ranked_doc_lists와 weights 길이가 달라야 합니다."

    scores = {}
    doc_map = {}

    for docs, weight in zip(ranked_doc_lists, weights):
        for rank, doc in enumerate(docs, start=1):
            doc_id = get_doc_id(doc)

            if doc_id not in scores:
                scores[doc_id] = 0.0
                doc_map[doc_id] = doc

            scores[doc_id] += weight * (1.0 / (rrf_k + rank))

    ranked_doc_ids = sorted(
        scores.keys(),
        key=lambda doc_id: scores[doc_id],
        reverse=True,
    )

    fused_docs = []

    for doc_id in ranked_doc_ids[:top_n]:
        doc = doc_map[doc_id]
        metadata = dict(doc.metadata)
        metadata["rrf_score"] = scores[doc_id]

        fused_docs.append(
            Document(
                page_content=doc.page_content,
                metadata=metadata,
            )
        )

    return fused_docs

In [33]:
def retrieve_hybrid_rrf(
    question: str,
    dense_k: int = 10,
    bm25_k: int = 10,
    final_k: int = 5,
    dense_weight: float = 0.5,
    bm25_weight: float = 0.5,
    rrf_k: int = 60,
) -> list[Document]:
    """
    C. Hybrid Search

    Dense retrieval과 BM25 retrieval 결과를 RRF로 결합한다.
    """
    dense_docs = retrieve_dense_only(question, k=dense_k)
    bm25_docs = retrieve_bm25_only(question, k=bm25_k)

    return reciprocal_rank_fusion(
        ranked_doc_lists=[dense_docs, bm25_docs],
        weights=[dense_weight, bm25_weight],
        rrf_k=rrf_k,
        top_n=final_k,
    )

In [34]:
test_question = "Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?"

hybrid_docs = retrieve_hybrid_rrf(
    question=test_question,
    dense_k=10,
    bm25_k=10,
    final_k=5,
)

hybrid_df = preview_docs(hybrid_docs)
hybrid_df["rrf_score"] = [doc.metadata.get("rrf_score") for doc in hybrid_docs]
hybrid_df.insert(0, "retriever", "hybrid_rrf")

display(hybrid_df)

,retriever,rank,source_file,game_key,section,chunk_index,preview,rrf_score
0,hybrid_rrf,1,monster_hunter_world.md,monster_hunter_world,metadata,46,## Metadata - game_key: monster_hunter_world -...,0.015772
1,hybrid_rrf,2,monster_hunter_world.md,monster_hunter_world,about,48,## About The Game Welcome to a new world! Take...,0.015625
2,hybrid_rrf,3,monster_hunter_world.md,monster_hunter_world,news,57,## Steam News and Updates ### News 1: Monster ...,0.015289
3,hybrid_rrf,4,monster_hunter_world.md,monster_hunter_world,news,59,{STEAM_CLAN_IMAGE}/45725708/73392ac8e0f10e408b...,0.008197
4,hybrid_rrf,5,monster_hunter_world.md,monster_hunter_world,store_summary,47,## Store Summary Welcome to a new world! In Mo...,0.008065


In [35]:
def compare_three_retrievers(question: str, k: int = 5) -> pd.DataFrame:
    dense_docs = retrieve_dense_only(question, k=k)
    bm25_docs = retrieve_bm25_only(question, k=k)
    hybrid_docs = retrieve_hybrid_rrf(
        question=question,
        dense_k=10,
        bm25_k=10,
        final_k=k,
    )

    rows = []

    for retriever_name, docs in [
        ("dense_only", dense_docs),
        ("bm25_only", bm25_docs),
        ("hybrid_rrf", hybrid_docs),
    ]:
        for rank, doc in enumerate(docs, start=1):
            rows.append(
                {
                    "question": question,
                    "retriever": retriever_name,
                    "rank": rank,
                    "source_file": doc.metadata.get("source_file"),
                    "game_key": doc.metadata.get("game_key"),
                    "section": doc.metadata.get("section"),
                    "chunk_index": doc.metadata.get("chunk_index"),
                    "rrf_score": doc.metadata.get("rrf_score"),
                    "preview": doc.page_content[:300].replace("\n", " "),
                }
            )

    return pd.DataFrame(rows)

test_question = "No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?"

display(compare_three_retrievers(test_question, k=5))

,question,retriever,rank,source_file,game_key,section,chunk_index,rrf_score,preview
0,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,dense_only,1,no_mans_sky.md,no_mans_sky,news,93,NaN,### News 4: No Man's Sky's latest update is a ...
1,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,dense_only,2,no_mans_sky.md,no_mans_sky,news,89,NaN,### News 3: Introducing No Man's Sky Xeno Aren...
2,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,dense_only,3,no_mans_sky.md,no_mans_sky,news,87,NaN,## Steam News and Updates ### News 1: No Man's...
3,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,dense_only,4,no_mans_sky.md,no_mans_sky,news,90,NaN,"Hello everyone,This is already shaping up to a..."
4,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,dense_only,5,no_mans_sky.md,no_mans_sky,store_summary,71,NaN,## Store Summary No Man's Sky is a game about ...
5,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,bm25_only,1,hollow_knight.md,hollow_knight,news,43,NaN,### News 2: Just in time for Hollow Knight's 9...
6,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,bm25_only,2,baldurs_gate_3.md,baldurs_gate_3,news,17,NaN,### News 3: Astarion's actor from Baldur's Gat...
7,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,bm25_only,3,no_mans_sky.md,no_mans_sky,news,87,NaN,## Steam News and Updates ### News 1: No Man's...
8,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,bm25_only,4,hollow_knight.md,hollow_knight,news,40,NaN,## Steam News and Updates ### News 1: Patch Ve...
9,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,bm25_only,5,no_mans_sky.md,no_mans_sky,news,93,NaN,### News 4: No Man's Sky's latest update is a ...


In [36]:
retrieval_compare_rows = []

for question in eval_questions:
    compare_df = compare_three_retrievers(question, k=5)
    retrieval_compare_rows.append(compare_df)

retrieval_compare_df = pd.concat(retrieval_compare_rows, ignore_index=True)

display(retrieval_compare_df)

retrieval_compare_path = EVAL_DIR / "week5_dense_bm25_hybrid_retrieval_results.csv"
retrieval_compare_df.to_csv(retrieval_compare_path, index=False, encoding="utf-8-sig")

print("saved:", retrieval_compare_path)

,question,retriever,rank,source_file,game_key,section,chunk_index,rrf_score,preview
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,dense_only,1,hollow_knight.md,hollow_knight,store_summary,31,NaN,## Store Summary Forge your own path in Hollow...
1,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,dense_only,2,hollow_knight.md,hollow_knight,about,35,NaN,Complete Hollow Knight to unlock Steel Soul Mo...
2,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,dense_only,3,hollow_knight.md,hollow_knight,metadata,30,NaN,## Metadata - game_key: hollow_knight - appid:...
3,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,dense_only,4,no_mans_sky.md,no_mans_sky,news,92,NaN,rewards. It's a genuine path for players to pr...
4,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,dense_only,5,hollow_knight.md,hollow_knight,about,32,NaN,## About The Game Hollow Knight Expands with F...
...,...,...,...,...,...,...,...,...,...
145,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,hybrid_rrf,1,cyberpunk_2077.md,cyberpunk_2077,store_summary,20,0.016001,## Store Summary Cyberpunk 2077 is an open-wor...
146,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,hybrid_rrf,2,cyberpunk_2077.md,cyberpunk_2077,about,21,0.015443,## About The Game Cyberpunk 2077: Ultimate Edi...
147,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,hybrid_rrf,3,cyberpunk_2077.md,cyberpunk_2077,metadata,19,0.015388,## Metadata - game_key: cyberpunk_2077 - appid...
148,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,hybrid_rrf,4,baldurs_gate_3.md,baldurs_gate_3,metadata,0,0.008197,## Metadata - game_key: baldurs_gate_3 - appid...


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_dense_bm25_hybrid_retrieval_results.csv


In [37]:
section_distribution_rows = []

for retriever_name, retrieve_fn in [
    ("dense_only", retrieve_dense_only),
    ("bm25_only", retrieve_bm25_only),
    ("hybrid_rrf", retrieve_hybrid_rrf),
]:
    for question in eval_questions:
        docs = retrieve_fn(question, k=5) if retriever_name != "hybrid_rrf" else retrieve_fn(question, final_k=5)

        section_counts = Counter(doc.metadata.get("section", "unknown") for doc in docs)
        game_counts = Counter(doc.metadata.get("game_key", "unknown") for doc in docs)

        section_distribution_rows.append(
            {
                "retriever": retriever_name,
                "question": question,
                "intent": infer_intent(question),
                "inferred_game_key": infer_game_key(question),
                "section_distribution": json.dumps(dict(section_counts), ensure_ascii=False),
                "game_distribution": json.dumps(dict(game_counts), ensure_ascii=False),
                "top1_game_key": docs[0].metadata.get("game_key") if docs else None,
                "top1_section": docs[0].metadata.get("section") if docs else None,
            }
        )

section_distribution_df = pd.DataFrame(section_distribution_rows)

display(section_distribution_df)

section_distribution_path = EVAL_DIR / "week5_retriever_section_distribution.csv"
section_distribution_df.to_csv(section_distribution_path, index=False, encoding="utf-8-sig")

print("saved:", section_distribution_path)

,retriever,question,intent,inferred_game_key,section_distribution,game_distribution,top1_game_key,top1_section
0,dense_only,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,gameplay,hollow_knight,"{""store_summary"": 1, ""about"": 2, ""metadata"": 1...","{""hollow_knight"": 4, ""no_mans_sky"": 1}",hollow_knight,store_summary
1,dense_only,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,gameplay,hollow_knight,"{""about"": 2, ""store_summary"": 1, ""news"": 2}","{""hollow_knight"": 5}",hollow_knight,about
2,dense_only,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,gameplay,monster_hunter_world,"{""news"": 3, ""store_summary"": 1, ""about"": 1}","{""monster_hunter_world"": 5}",monster_hunter_world,news
3,dense_only,Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?,gameplay,monster_hunter_world,"{""about"": 2, ""store_summary"": 1, ""news"": 2}","{""monster_hunter_world"": 3, ""no_mans_sky"": 1, ...",monster_hunter_world,about
4,dense_only,Baldur's Gate 3는 어떤 RPG인가요?,gameplay,baldurs_gate_3,"{""store_summary"": 1, ""review"": 1, ""metadata"": ...","{""baldurs_gate_3"": 4, ""cyberpunk_2077"": 1}",baldurs_gate_3,store_summary
5,dense_only,Baldur's Gate 3에서 선택과 서사는 어떤 역할을 하나요?,gameplay,baldurs_gate_3,"{""store_summary"": 1, ""news"": 2, ""about"": 1, ""m...","{""baldurs_gate_3"": 5}",baldurs_gate_3,store_summary
6,dense_only,No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?,news,no_mans_sky,"{""news"": 4, ""store_summary"": 1}","{""no_mans_sky"": 5}",no_mans_sky,news
7,dense_only,No Man's Sky의 최근 뉴스나 업데이트 방향은 무엇인가요?,news,no_mans_sky,"{""news"": 5}","{""no_mans_sky"": 4, ""monster_hunter_world"": 1}",no_mans_sky,news
8,dense_only,Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?,review,cyberpunk_2077,"{""news"": 1, ""about"": 1, ""store_summary"": 1, ""r...","{""cyberpunk_2077"": 4, ""no_mans_sky"": 1}",cyberpunk_2077,news
9,dense_only,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,gameplay,cyberpunk_2077,"{""about"": 1, ""store_summary"": 1, ""news"": 1, ""m...","{""cyberpunk_2077"": 4, ""no_mans_sky"": 1}",cyberpunk_2077,about


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_retriever_section_distribution.csv


In [38]:
def measure_latency(retrieve_fn, question: str, **kwargs):
    start = time.perf_counter()
    docs = retrieve_fn(question, **kwargs)
    end = time.perf_counter()

    return docs, end - start

In [39]:
latency_rows = []

for question in eval_questions:
    _, dense_latency = measure_latency(
        retrieve_dense_only,
        question,
        k=5,
    )

    _, bm25_latency = measure_latency(
        retrieve_bm25_only,
        question,
        k=5,
    )

    _, hybrid_latency = measure_latency(
        retrieve_hybrid_rrf,
        question,
        dense_k=10,
        bm25_k=10,
        final_k=5,
    )

    latency_rows.extend(
        [
            {
                "question": question,
                "retriever": "dense_only",
                "latency_s": dense_latency,
            },
            {
                "question": question,
                "retriever": "bm25_only",
                "latency_s": bm25_latency,
            },
            {
                "question": question,
                "retriever": "hybrid_rrf",
                "latency_s": hybrid_latency,
            },
        ]
    )

latency_df = pd.DataFrame(latency_rows)

display(latency_df)

latency_summary_df = (
    latency_df
    .groupby("retriever", as_index=False)
    .agg(
        avg_latency_s=("latency_s", "mean"),
        min_latency_s=("latency_s", "min"),
        max_latency_s=("latency_s", "max"),
    )
)

display(latency_summary_df)

latency_path = EVAL_DIR / "week5_dense_bm25_hybrid_latency.csv"
latency_summary_path = EVAL_DIR / "week5_dense_bm25_hybrid_latency_summary.csv"

latency_df.to_csv(latency_path, index=False, encoding="utf-8-sig")
latency_summary_df.to_csv(latency_summary_path, index=False, encoding="utf-8-sig")

print("saved:", latency_path)
print("saved:", latency_summary_path)

,question,retriever,latency_s
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,dense_only,0.029139
1,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,bm25_only,0.000922
2,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,hybrid_rrf,0.027993
3,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,dense_only,0.039953
4,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,bm25_only,0.000873
5,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,hybrid_rrf,0.036426
6,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,dense_only,0.030046
7,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,bm25_only,0.000496
8,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,hybrid_rrf,0.027609
9,Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?,dense_only,0.025430


,retriever,avg_latency_s,min_latency_s,max_latency_s
0,bm25_only,0.000581,0.000399,0.000922
1,dense_only,0.028445,0.023855,0.039953
2,hybrid_rrf,0.028821,0.025669,0.036426


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_dense_bm25_hybrid_latency.csv
saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_dense_bm25_hybrid_latency_summary.csv


In [40]:
RRF_WEIGHT_CONFIGS = [
    {"config_name": "bm25_0.3_dense_0.7", "bm25_weight": 0.3, "dense_weight": 0.7},
    {"config_name": "bm25_0.5_dense_0.5", "bm25_weight": 0.5, "dense_weight": 0.5},
    {"config_name": "bm25_0.7_dense_0.3", "bm25_weight": 0.7, "dense_weight": 0.3},
]

rrf_tuning_rows = []

for config in RRF_WEIGHT_CONFIGS:
    for question in eval_questions:
        docs = retrieve_hybrid_rrf(
            question=question,
            dense_k=10,
            bm25_k=10,
            final_k=5,
            dense_weight=config["dense_weight"],
            bm25_weight=config["bm25_weight"],
        )

        for rank, doc in enumerate(docs, start=1):
            rrf_tuning_rows.append(
                {
                    "config_name": config["config_name"],
                    "question": question,
                    "rank": rank,
                    "source_file": doc.metadata.get("source_file"),
                    "game_key": doc.metadata.get("game_key"),
                    "section": doc.metadata.get("section"),
                    "chunk_index": doc.metadata.get("chunk_index"),
                    "rrf_score": doc.metadata.get("rrf_score"),
                    "preview": doc.page_content[:250].replace("\n", " "),
                }
            )

rrf_tuning_df = pd.DataFrame(rrf_tuning_rows)

display(rrf_tuning_df)

rrf_tuning_path = EVAL_DIR / "week5_rrf_weight_tuning_results.csv"
rrf_tuning_df.to_csv(rrf_tuning_path, index=False, encoding="utf-8-sig")

print("saved:", rrf_tuning_path)

,config_name,question,rank,source_file,game_key,section,chunk_index,rrf_score,preview
0,bm25_0.3_dense_0.7,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,1,hollow_knight.md,hollow_knight,about,35,0.016052,Complete Hollow Knight to unlock Steel Soul Mo...
1,bm25_0.3_dense_0.7,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,2,hollow_knight.md,hollow_knight,store_summary,31,0.016021,## Store Summary Forge your own path in Hollow...
2,bm25_0.3_dense_0.7,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,3,hollow_knight.md,hollow_knight,metadata,30,0.015459,## Metadata - game_key: hollow_knight - appid:...
3,bm25_0.3_dense_0.7,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,4,no_mans_sky.md,no_mans_sky,news,92,0.010937,rewards. It's a genuine path for players to pr...
4,bm25_0.3_dense_0.7,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,5,hollow_knight.md,hollow_knight,about,32,0.010769,## About The Game Hollow Knight Expands with F...
...,...,...,...,...,...,...,...,...,...
145,bm25_0.7_dense_0.3,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,1,cyberpunk_2077.md,cyberpunk_2077,store_summary,20,0.015950,## Store Summary Cyberpunk 2077 is an open-wor...
146,bm25_0.7_dense_0.3,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,2,cyberpunk_2077.md,cyberpunk_2077,metadata,19,0.015294,## Metadata - game_key: cyberpunk_2077 - appid...
147,bm25_0.7_dense_0.3,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,3,cyberpunk_2077.md,cyberpunk_2077,about,21,0.015063,## About The Game Cyberpunk 2077: Ultimate Edi...
148,bm25_0.7_dense_0.3,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,4,baldurs_gate_3.md,baldurs_gate_3,metadata,0,0.011475,## Metadata - game_key: baldurs_gate_3 - appid...


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_rrf_weight_tuning_results.csv


In [41]:
try:
    import torch

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_FP16 = torch.cuda.is_available()

    print("torch version:", torch.__version__)
    print("device:", DEVICE)
    print("use_fp16:", USE_FP16)

except Exception as e:
    DEVICE = "cpu"
    USE_FP16 = False

    print("torch 확인 실패. CPU 모드로 진행합니다.")
    print(type(e).__name__, e)

torch version: 2.8.0+cu126
device: cuda
use_fp16: True


In [42]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_NAME)
reranker_model = AutoModelForSequenceClassification.from_pretrained(RERANKER_MODEL_NAME)

reranker_model.to(DEVICE)
reranker_model.eval()

if DEVICE == "cuda":
    reranker_model.half()

print("Reranker loaded with Transformers:", RERANKER_MODEL_NAME)
print("device:", DEVICE)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranker loaded with Transformers: BAAI/bge-reranker-v2-m3
device: cuda


In [43]:
def compute_reranker_scores(
    question: str,
    docs: list[Document],
    batch_size: int = 8,
    max_length: int = 1024,
    normalize: bool = False,
) -> list[float]:
    """
    Transformers 직접 로드 방식의 BGE Cross-Encoder reranker score 계산.

    입력:
    - question: 사용자 질문
    - docs: Hybrid RRF로 가져온 후보 chunk 리스트

    출력:
    - 각 chunk에 대한 relevance score
    """
    if not docs:
        return []

    all_scores = []

    for start_idx in range(0, len(docs), batch_size):
        batch_docs = docs[start_idx:start_idx + batch_size]
        pairs = [[question, doc.page_content] for doc in batch_docs]

        with torch.no_grad():
            inputs = reranker_tokenizer(
                pairs,
                padding=True,
                truncation=True,
                return_tensors="pt",
                max_length=max_length,
            )

            inputs = {
                key: value.to(DEVICE)
                for key, value in inputs.items()
            }

            outputs = reranker_model(
                **inputs,
                return_dict=True,
            )

            scores = outputs.logits.view(-1).float()

            if normalize:
                scores = torch.sigmoid(scores)

            scores = scores.detach().cpu().numpy().tolist()
            all_scores.extend(float(score) for score in scores)

    return all_scores

In [44]:
def rerank_documents(
    question: str,
    docs: list[Document],
    top_n: int = 5,
) -> list[Document]:
    """
    후보 문서들을 BGE Cross-Encoder reranker score 기준으로 재정렬한다.
    """
    scores = compute_reranker_scores(
        question=question,
        docs=docs,
        batch_size=8,
        max_length=1024,
        normalize=False,
    )

    scored_docs = []

    for original_rank, (doc, score) in enumerate(zip(docs, scores), start=1):
        metadata = dict(doc.metadata)
        metadata["reranker_model"] = RERANKER_MODEL_NAME
        metadata["reranker_score"] = score
        metadata["pre_rerank_rank"] = original_rank

        scored_docs.append(
            Document(
                page_content=doc.page_content,
                metadata=metadata,
            )
        )

    scored_docs = sorted(
        scored_docs,
        key=lambda doc: doc.metadata.get("reranker_score", float("-inf")),
        reverse=True,
    )

    reranked_docs = []

    for rerank_rank, doc in enumerate(scored_docs[:top_n], start=1):
        metadata = dict(doc.metadata)
        metadata["rerank_rank"] = rerank_rank

        reranked_docs.append(
            Document(
                page_content=doc.page_content,
                metadata=metadata,
            )
        )

    return reranked_docs

In [45]:
def retrieve_hybrid_rerank(
    question: str,
    first_k: int = 20,
    final_k: int = 5,
    dense_weight: float = 0.5,
    bm25_weight: float = 0.5,
) -> list[Document]:
    """
    D. Hybrid + BGE Cross-Encoder Re-ranker

    1. Dense top-k 검색
    2. BM25 top-k 검색
    3. RRF로 후보 first_k개 결합
    4. BGE Cross-Encoder Re-ranker로 final_k개 재정렬
    """
    hybrid_candidates = retrieve_hybrid_rrf(
        question=question,
        dense_k=first_k,
        bm25_k=first_k,
        final_k=first_k,
        dense_weight=dense_weight,
        bm25_weight=bm25_weight,
    )

    reranked_docs = rerank_documents(
        question=question,
        docs=hybrid_candidates,
        top_n=final_k,
    )

    return reranked_docs

In [46]:
test_question = "Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?"

hybrid_before_docs = retrieve_hybrid_rrf(
    question=test_question,
    dense_k=20,
    bm25_k=20,
    final_k=20,
)

hybrid_after_docs = retrieve_hybrid_rerank(
    question=test_question,
    first_k=20,
    final_k=5,
)

before_df = preview_docs(hybrid_before_docs, max_chars=250)
before_df["rrf_score"] = [doc.metadata.get("rrf_score") for doc in hybrid_before_docs]
before_df.insert(0, "retriever", "hybrid_before_rerank")

after_df = preview_docs(hybrid_after_docs, max_chars=250)
after_df["rrf_score"] = [doc.metadata.get("rrf_score") for doc in hybrid_after_docs]
after_df["pre_rerank_rank"] = [doc.metadata.get("pre_rerank_rank") for doc in hybrid_after_docs]
after_df["reranker_score"] = [doc.metadata.get("reranker_score") for doc in hybrid_after_docs]
after_df["rerank_rank"] = [doc.metadata.get("rerank_rank") for doc in hybrid_after_docs]
after_df.insert(0, "retriever", "hybrid_after_rerank")

display(before_df.head(10))
display(after_df)

,retriever,rank,source_file,game_key,section,chunk_index,preview,rrf_score
0,hybrid_before_rerank,1,monster_hunter_world.md,monster_hunter_world,metadata,46,## Metadata - game_key: monster_hunter_world -...,0.015772
1,hybrid_before_rerank,2,monster_hunter_world.md,monster_hunter_world,about,48,## About The Game Welcome to a new world! Take...,0.015625
2,hybrid_before_rerank,3,monster_hunter_world.md,monster_hunter_world,news,57,## Steam News and Updates ### News 1: Monster ...,0.015289
3,hybrid_before_rerank,4,monster_hunter_world.md,monster_hunter_world,store_summary,47,## Store Summary Welcome to a new world! In Mo...,0.015009
4,hybrid_before_rerank,5,monster_hunter_world.md,monster_hunter_world,news,59,{STEAM_CLAN_IMAGE}/45725708/73392ac8e0f10e408b...,0.014776
5,hybrid_before_rerank,6,monster_hunter_world.md,monster_hunter_world,news,65,The third entry in the Monster Hunter Stories ...,0.014425
6,hybrid_before_rerank,7,hollow_knight.md,hollow_knight,about,35,Complete Hollow Knight to unlock Steel Soul Mo...,0.014003
7,hybrid_before_rerank,8,monster_hunter_world.md,monster_hunter_world,news,64,### News 4: Pre-Order Monster Hunter Stories 3...,0.013942
8,hybrid_before_rerank,9,monster_hunter_world.md,monster_hunter_world,news,63,portion does not include news about anything o...,0.013873
9,hybrid_before_rerank,10,monster_hunter_world.md,monster_hunter_world,review,53,really fun peak game i played rise before worl...,0.013722


,retriever,rank,source_file,game_key,section,chunk_index,preview,rrf_score,pre_rerank_rank,reranker_score,rerank_rank
0,hybrid_after_rerank,1,monster_hunter_world.md,monster_hunter_world,metadata,46,## Metadata - game_key: monster_hunter_world -...,0.015772,1,-4.234375,1
1,hybrid_after_rerank,2,monster_hunter_world.md,monster_hunter_world,about,48,## About The Game Welcome to a new world! Take...,0.015625,2,-4.484375,2
2,hybrid_after_rerank,3,monster_hunter_world.md,monster_hunter_world,review,53,really fun peak game i played rise before worl...,0.013722,10,-4.554688,3
3,hybrid_after_rerank,4,monster_hunter_world.md,monster_hunter_world,news,57,## Steam News and Updates ### News 1: Monster ...,0.015289,3,-5.183594,4
4,hybrid_after_rerank,5,monster_hunter_world.md,monster_hunter_world,store_summary,47,## Store Summary Welcome to a new world! In Mo...,0.015009,4,-5.421875,5


In [47]:
retriever_registry = {
    "dense_only": lambda question, k=5: retrieve_dense_only(
        question=question,
        k=k,
    ),

    "bm25_only": lambda question, k=5: retrieve_bm25_only(
        question=question,
        k=k,
    ),

    "hybrid_rrf": lambda question, k=5: retrieve_hybrid_rrf(
        question=question,
        dense_k=20,
        bm25_k=20,
        final_k=k,
        dense_weight=0.7,
        bm25_weight=0.3,
    ),

    "hybrid_rerank": lambda question, k=5: retrieve_hybrid_rerank(
        question=question,
        first_k=20,
        final_k=k,
        dense_weight=0.7,
        bm25_weight=0.3,
    ),
}

print("registered retrievers:")
for name in retriever_registry.keys():
    print("-", name)

registered retrievers:
- dense_only
- bm25_only
- hybrid_rrf
- hybrid_rerank


In [48]:
test_question = "Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?"

registry_test_rows = []

for retriever_name, retrieve_fn in retriever_registry.items():
    docs = retrieve_fn(test_question, k=5)

    for rank, doc in enumerate(docs, start=1):
        registry_test_rows.append(
            {
                "retriever": retriever_name,
                "rank": rank,
                "source_file": doc.metadata.get("source_file"),
                "game_key": doc.metadata.get("game_key"),
                "section": doc.metadata.get("section"),
                "chunk_index": doc.metadata.get("chunk_index"),
                "rrf_score": doc.metadata.get("rrf_score"),
                "pre_rerank_rank": doc.metadata.get("pre_rerank_rank"),
                "reranker_score": doc.metadata.get("reranker_score"),
                "preview": doc.page_content[:250].replace("\n", " "),
            }
        )

registry_test_df = pd.DataFrame(registry_test_rows)
display(registry_test_df)

,retriever,rank,source_file,game_key,section,chunk_index,rrf_score,pre_rerank_rank,reranker_score,preview
0,dense_only,1,monster_hunter_world.md,monster_hunter_world,news,59,NaN,NaN,NaN,{STEAM_CLAN_IMAGE}/45725708/73392ac8e0f10e408b...
1,dense_only,2,monster_hunter_world.md,monster_hunter_world,store_summary,47,NaN,NaN,NaN,## Store Summary Welcome to a new world! In Mo...
2,dense_only,3,monster_hunter_world.md,monster_hunter_world,news,57,NaN,NaN,NaN,## Steam News and Updates ### News 1: Monster ...
3,dense_only,4,monster_hunter_world.md,monster_hunter_world,about,48,NaN,NaN,NaN,## About The Game Welcome to a new world! Take...
4,dense_only,5,monster_hunter_world.md,monster_hunter_world,news,64,NaN,NaN,NaN,### News 4: Pre-Order Monster Hunter Stories 3...
5,bm25_only,1,monster_hunter_world.md,monster_hunter_world,metadata,46,NaN,NaN,NaN,## Metadata - game_key: monster_hunter_world -...
6,bm25_only,2,baldurs_gate_3.md,baldurs_gate_3,metadata,0,NaN,NaN,NaN,## Metadata - game_key: baldurs_gate_3 - appid...
7,bm25_only,3,no_mans_sky.md,no_mans_sky,metadata,70,NaN,NaN,NaN,## Metadata - game_key: no_mans_sky - appid: 2...
8,bm25_only,4,monster_hunter_world.md,monster_hunter_world,about,48,NaN,NaN,NaN,## About The Game Welcome to a new world! Take...
9,bm25_only,5,baldurs_gate_3.md,baldurs_gate_3,about,4,NaN,NaN,NaN,"The Forgotten Realms are a vast, detailed, and..."


In [49]:
four_retriever_rows = []

for question in eval_questions:
    for retriever_name, retrieve_fn in retriever_registry.items():
        docs = retrieve_fn(question, k=5)

        for rank, doc in enumerate(docs, start=1):
            four_retriever_rows.append(
                {
                    "question": question,
                    "retriever": retriever_name,
                    "rank": rank,
                    "source_file": doc.metadata.get("source_file"),
                    "game_key": doc.metadata.get("game_key"),
                    "section": doc.metadata.get("section"),
                    "chunk_index": doc.metadata.get("chunk_index"),
                    "rrf_score": doc.metadata.get("rrf_score"),
                    "pre_rerank_rank": doc.metadata.get("pre_rerank_rank"),
                    "reranker_score": doc.metadata.get("reranker_score"),
                    "preview": doc.page_content[:300].replace("\n", " "),
                }
            )

four_retriever_df = pd.DataFrame(four_retriever_rows)

display(four_retriever_df)

four_retriever_path = EVAL_DIR / "week5_four_retriever_results.csv"
four_retriever_df.to_csv(four_retriever_path, index=False, encoding="utf-8-sig")

print("saved:", four_retriever_path)

,question,retriever,rank,source_file,game_key,section,chunk_index,rrf_score,pre_rerank_rank,reranker_score,preview
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,dense_only,1,hollow_knight.md,hollow_knight,store_summary,31,NaN,NaN,NaN,## Store Summary Forge your own path in Hollow...
1,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,dense_only,2,hollow_knight.md,hollow_knight,about,35,NaN,NaN,NaN,Complete Hollow Knight to unlock Steel Soul Mo...
2,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,dense_only,3,hollow_knight.md,hollow_knight,metadata,30,NaN,NaN,NaN,## Metadata - game_key: hollow_knight - appid:...
3,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,dense_only,4,no_mans_sky.md,no_mans_sky,news,92,NaN,NaN,NaN,rewards. It's a genuine path for players to pr...
4,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,dense_only,5,hollow_knight.md,hollow_knight,about,32,NaN,NaN,NaN,## About The Game Hollow Knight Expands with F...
...,...,...,...,...,...,...,...,...,...,...,...
195,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,hybrid_rerank,1,cyberpunk_2077.md,cyberpunk_2077,store_summary,20,0.016052,1.0,0.398682,## Store Summary Cyberpunk 2077 is an open-wor...
196,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,hybrid_rerank,2,cyberpunk_2077.md,cyberpunk_2077,metadata,19,0.015483,3.0,-1.353516,## Metadata - game_key: cyberpunk_2077 - appid...
197,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,hybrid_rerank,3,cyberpunk_2077.md,cyberpunk_2077,review,24,0.013745,7.0,-3.035156,Its fun ### Review 11 - sentiment: positive -...
198,Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?,hybrid_rerank,4,cyberpunk_2077.md,cyberpunk_2077,news,27,0.015278,4.0,-4.027344,## Steam News and Updates ### News 1: Cyberpun...


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_four_retriever_results.csv


In [50]:
four_latency_rows = []

for question in eval_questions:
    for retriever_name, retrieve_fn in retriever_registry.items():
        start = time.perf_counter()
        docs = retrieve_fn(question, k=5)
        end = time.perf_counter()

        four_latency_rows.append(
            {
                "question": question,
                "retriever": retriever_name,
                "latency_s": end - start,
                "returned_docs": len(docs),
            }
        )

four_latency_df = pd.DataFrame(four_latency_rows)

four_latency_summary_df = (
    four_latency_df
    .groupby("retriever", as_index=False)
    .agg(
        avg_latency_s=("latency_s", "mean"),
        min_latency_s=("latency_s", "min"),
        max_latency_s=("latency_s", "max"),
    )
    .sort_values("avg_latency_s")
)

display(four_latency_df)
display(four_latency_summary_df)

four_latency_path = EVAL_DIR / "week5_four_retriever_latency.csv"
four_latency_summary_path = EVAL_DIR / "week5_four_retriever_latency_summary.csv"

four_latency_df.to_csv(four_latency_path, index=False, encoding="utf-8-sig")
four_latency_summary_df.to_csv(four_latency_summary_path, index=False, encoding="utf-8-sig")

print("saved:", four_latency_path)
print("saved:", four_latency_summary_path)

,question,retriever,latency_s,returned_docs
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,dense_only,0.039839,5
1,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,bm25_only,0.000646,5
2,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,hybrid_rrf,0.030380,5
3,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,hybrid_rerank,0.267796,5
4,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,dense_only,0.027083,5
5,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,bm25_only,0.001100,5
6,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,hybrid_rrf,0.030495,5
7,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,hybrid_rerank,0.279163,5
8,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,dense_only,0.045579,5
9,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,bm25_only,0.000452,5


,retriever,avg_latency_s,min_latency_s,max_latency_s
0,bm25_only,0.000706,0.000405,0.001157
3,hybrid_rrf,0.029281,0.021535,0.036457
1,dense_only,0.031188,0.021133,0.054430
2,hybrid_rerank,0.276223,0.245068,0.310273


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_four_retriever_latency.csv
saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_four_retriever_latency_summary.csv


In [51]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from datasets import Dataset

ANSWER_MODEL_NAME = "gpt-5-mini"

answer_llm = ChatOpenAI(
    model=ANSWER_MODEL_NAME,
    temperature=1,
)

print("Answer LLM:", ANSWER_MODEL_NAME)

Answer LLM: gpt-5-mini


In [52]:
rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a Steam game recommendation and analysis assistant.

Answer the user's question using only the provided context.
If the context is insufficient, say that the available documents do not contain enough evidence.
Do not invent details that are not supported by the context.

Write the answer in Korean.
Keep the answer concise but grounded.
            """.strip(),
        ),
        (
            "human",
            """
[Question]
{question}

[Context]
{context}
            """.strip(),
        ),
    ]
)

In [53]:
def format_docs_for_context(docs: list[Document]) -> str:
    formatted = []

    for i, doc in enumerate(docs, start=1):
        source_file = doc.metadata.get("source_file", "unknown")
        game_key = doc.metadata.get("game_key", "unknown")
        section = doc.metadata.get("section", "unknown")
        chunk_index = doc.metadata.get("chunk_index", "unknown")
        content = doc.page_content

        formatted.append(
            f"[Context {i}]\n"
            f"source_file: {source_file}\n"
            f"game_key: {game_key}\n"
            f"section: {section}\n"
            f"chunk_index: {chunk_index}\n"
            f"{content}"
        )

    return "\n\n".join(formatted)

In [54]:
def generate_rag_answer(question: str, docs: list[Document]) -> str:
    context = format_docs_for_context(docs)

    chain = rag_prompt | answer_llm

    response = chain.invoke(
        {
            "question": question,
            "context": context,
        }
    )

    return response.content

In [55]:
reference_answers = {
    "Hollow Knight는 어떤 플레이 스타일의 게임인가요?":
        "Hollow Knight는 2D 액션 어드벤처/메트로배니아 스타일의 게임으로, 탐험, 전투, 플랫폼 액션, 능력 획득을 통한 지역 확장이 핵심이다.",

    "Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?":
        "Hollow Knight는 Hallownest라는 거대한 지하 세계를 배경으로 하며, 어둡고 신비로운 분위기, 연결된 지역 탐험, 숨겨진 비밀 발견이 특징이다.",

    "Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?":
        "Monster Hunter: World의 핵심 플레이 루프는 몬스터를 추적하고 사냥한 뒤, 획득한 소재로 장비를 제작·강화하고 더 강한 몬스터에 도전하는 구조이다.",

    "Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?":
        "Monster Hunter: World는 여러 플레이어가 함께 몬스터를 사냥하는 협동 플레이를 지원하며, 역할 분담과 장비 조합을 통해 사냥을 진행하는 것이 특징이다.",

    "Baldur's Gate 3는 어떤 RPG인가요?":
        "Baldur's Gate 3는 Dungeons & Dragons 규칙을 기반으로 한 파티 기반 판타지 RPG이며, 턴제 전투, 캐릭터 빌드, 선택 중심 진행이 특징이다.",

    "Baldur's Gate 3에서 선택과 서사는 어떤 역할을 하나요?":
        "Baldur's Gate 3에서는 플레이어의 선택이 대화, 퀘스트, 동료 관계, 전투 결과와 이야기 전개에 영향을 주며, 선택과 서사가 핵심 경험을 이룬다.",

    "No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?":
        "No Man's Sky는 출시 이후 여러 업데이트를 통해 콘텐츠, 탐험 요소, 시스템, 편의 기능을 지속적으로 확장해 온 게임이다.",

    "No Man's Sky의 최근 뉴스나 업데이트 방향은 무엇인가요?":
        "No Man's Sky의 최근 뉴스와 업데이트는 새로운 콘텐츠, 이벤트, 탐험 요소, 시스템 개선 등 지속적인 확장과 개선 방향을 보여준다.",

    "Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?":
        "Cyberpunk 2077의 최근 리뷰는 게임의 세계관, 스토리, 그래픽, 몰입감에 대한 긍정 반응과 함께 성능이나 버그 경험에 대한 언급이 섞일 수 있다.",

    "Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?":
        "Cyberpunk 2077은 미래 도시 Night City를 배경으로 한 오픈월드 액션 RPG이며, 스토리, 선택, 캐릭터 성장, 전투, 탐험이 주요 특징이다.",
}

In [77]:
RUN_FULL_EVAL = True

questions_for_eval = eval_questions if RUN_FULL_EVAL else eval_questions[:3]

print("평가 질문 수:", len(questions_for_eval))

rag_eval_rows = []

for retriever_name, retrieve_fn in retriever_registry.items():
    print(f"\n=== {retriever_name} ===")

    for question in questions_for_eval:
        start = time.perf_counter()

        docs = retrieve_fn(question, k=5)
        answer = generate_rag_answer(question, docs)

        end = time.perf_counter()

        rag_eval_rows.append(
            {
                "retriever": retriever_name,
                "question": question,
                "answer": answer,
                "reference": reference_answers[question],
                "contexts": [doc.page_content for doc in docs],
                "context_metadata": [
                    {
                        "source_file": doc.metadata.get("source_file"),
                        "game_key": doc.metadata.get("game_key"),
                        "section": doc.metadata.get("section"),
                        "chunk_index": doc.metadata.get("chunk_index"),
                        "rrf_score": doc.metadata.get("rrf_score"),
                        "pre_rerank_rank": doc.metadata.get("pre_rerank_rank"),
                        "reranker_score": doc.metadata.get("reranker_score"),
                    }
                    for doc in docs
                ],
                "latency_s": end - start,
            }
        )

        print(f"- done: {question[:40]}...")

rag_eval_df = pd.DataFrame(rag_eval_rows)

display(rag_eval_df[["retriever", "question", "answer", "latency_s"]].head())

rag_eval_path = EVAL_DIR / "week5_rag_answers_for_ragas.csv"
rag_eval_df.to_csv(rag_eval_path, index=False, encoding="utf-8-sig")

print("saved:", rag_eval_path)

평가 질문 수: 10

=== dense_only ===
- done: Hollow Knight는 어떤 플레이 스타일의 게임인가요?...
- done: Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?...
- done: Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?...
- done: Monster Hunter: World는 협동 플레이 측면에서 어떤 특징...
- done: Baldur's Gate 3는 어떤 RPG인가요?...
- done: Baldur's Gate 3에서 선택과 서사는 어떤 역할을 하나요?...
- done: No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?...
- done: No Man's Sky의 최근 뉴스나 업데이트 방향은 무엇인가요?...
- done: Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?...
- done: Cyberpunk 2077 문서에서 확인되는 주요 특징은 무엇인가요?...

=== bm25_only ===
- done: Hollow Knight는 어떤 플레이 스타일의 게임인가요?...
- done: Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?...
- done: Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?...
- done: Monster Hunter: World는 협동 플레이 측면에서 어떤 특징...
- done: Baldur's Gate 3는 어떤 RPG인가요?...
- done: Baldur's Gate 3에서 선택과 서사는 어떤 역할을 하나요?...
- done: No Man's Sky는 업데이트와 관련해서 어떤 내용이 있나요?...
- done: No Man's Sky의 최근 뉴스나 업데이트 방향은 무엇인가요?...
- done: Cyberpunk 2077 최근 리뷰에서는 어떤 반응이 있나요?...
- done: Cyberpunk 2077 문서에서 확인되는 주요 특징은

,retriever,question,answer,latency_s
0,dense_only,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,Hollow Knight는 클래식한 2D 핸드드로우 스타일의 액션 어드벤처 게임입니...,13.580360
1,dense_only,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,Hollow Knight의 분위기와 월드는 전반적으로 어둡고 서정적인 고딕풍 분위기...,6.460596
2,dense_only,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,간단히 말하면 반복되는 핵심 루프는 다음과 같습니다.\n\n- 퀘스트를 받고 다양한...,6.136432
3,dense_only,Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?,"간단히 요약하면 다음과 같습니다.\n\n- 혼자 플레이 가능하며, 협동 플레이는 최...",7.967831
4,dense_only,Baldur's Gate 3는 어떤 RPG인가요?,Baldur’s Gate 3는 던전 앤 드래곤즈 세계관을 바탕으로 한 스토리 중심의...,7.888234


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_rag_answers_for_ragas.csv


In [56]:
import ast
import pandas as pd

def parse_list_column(value, default=None):
    """
    CSV에 문자열로 저장된 list/dict 형태 컬럼을 다시 Python 객체로 복원한다.
    예: "['context1', 'context2']" -> ['context1', 'context2']
    """
    if default is None:
        default = []

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return default

    if isinstance(value, str):
        value = value.strip()

        if value == "":
            return default

        try:
            parsed = ast.literal_eval(value)
        except Exception:
            # contexts가 단일 문자열로 저장된 경우 fallback
            return [value]

        if isinstance(parsed, list):
            return parsed

        return [str(parsed)]

    return default


# 이전 셀을 실행하지 않고 저장된 CSV에서 바로 로드
rag_eval_path = EVAL_DIR / "week5_rag_answers_for_ragas.csv"

rag_eval_df = pd.read_csv(rag_eval_path)

print("loaded:", rag_eval_path)
print("shape:", rag_eval_df.shape)
print("columns:", rag_eval_df.columns.tolist())


# contexts 컬럼 복원
rag_eval_df["contexts"] = rag_eval_df["contexts"].apply(
    lambda x: [str(item) for item in parse_list_column(x, default=[])]
)

# context_metadata도 이후 Error Case 분석에서 쓸 수 있으므로 같이 복원
if "context_metadata" in rag_eval_df.columns:
    rag_eval_df["context_metadata"] = rag_eval_df["context_metadata"].apply(
        lambda x: parse_list_column(x, default=[])
    )


# reference 컬럼이 없고 ground_truth 컬럼만 있는 경우 대응
if "reference" not in rag_eval_df.columns and "ground_truth" in rag_eval_df.columns:
    rag_eval_df["reference"] = rag_eval_df["ground_truth"]


required_columns = ["question", "answer", "contexts", "reference", "retriever"]
missing_columns = [col for col in required_columns if col not in rag_eval_df.columns]

if missing_columns:
    raise ValueError(f"필수 컬럼이 없습니다: {missing_columns}")


ragas_dataset_rows = []

for _, row in rag_eval_df.iterrows():
    ragas_dataset_rows.append(
        {
            "question": row["question"],
            "answer": row["answer"],
            "contexts": row["contexts"],
            "ground_truth": row["reference"],
            "retriever": row["retriever"],
        }
    )

ragas_input_df = pd.DataFrame(ragas_dataset_rows)

display(ragas_input_df.head())

ragas_input_path = EVAL_DIR / "week5_ragas_input.csv"
ragas_input_df.to_csv(ragas_input_path, index=False, encoding="utf-8-sig")

print("saved:", ragas_input_path)
print("contexts type example:", type(ragas_input_df.loc[0, "contexts"]))
print("contexts length example:", len(ragas_input_df.loc[0, "contexts"]))

loaded: C:\Users\asguug\Documents\rag-agent\data\eval\week5_rag_answers_for_ragas.csv
shape: (40, 7)
columns: ['retriever', 'question', 'answer', 'reference', 'contexts', 'context_metadata', 'latency_s']


,question,answer,contexts,ground_truth,retriever
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,Hollow Knight는 클래식한 2D 핸드드로우 스타일의 액션 어드벤처 게임입니...,[## Store Summary\nForge your own path in Holl...,"Hollow Knight는 2D 액션 어드벤처/메트로배니아 스타일의 게임으로, 탐험...",dense_only
1,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,Hollow Knight의 분위기와 월드는 전반적으로 어둡고 서정적인 고딕풍 분위기...,[Complete Hollow Knight to unlock Steel Soul M...,Hollow Knight는 Hallownest라는 거대한 지하 세계를 배경으로 하며...,dense_only
2,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,간단히 말하면 반복되는 핵심 루프는 다음과 같습니다.\n\n- 퀘스트를 받고 다양한...,[{STEAM_CLAN_IMAGE}/45725708/73392ac8e0f10e408...,Monster Hunter: World의 핵심 플레이 루프는 몬스터를 추적하고 사냥...,dense_only
3,Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?,"간단히 요약하면 다음과 같습니다.\n\n- 혼자 플레이 가능하며, 협동 플레이는 최...",[## About The Game\nWelcome to a new world! Ta...,Monster Hunter: World는 여러 플레이어가 함께 몬스터를 사냥하는 협...,dense_only
4,Baldur's Gate 3는 어떤 RPG인가요?,Baldur’s Gate 3는 던전 앤 드래곤즈 세계관을 바탕으로 한 스토리 중심의...,[## Store Summary\nBaldur’s Gate 3 is a story-...,Baldur's Gate 3는 Dungeons & Dragons 규칙을 기반으로 한...,dense_only


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_ragas_input.csv
contexts type example: <class 'list'>
contexts length example: 5


In [57]:
import sys
import types

# RAGAS 0.4.3 compatibility patch for newer LangChain / VertexAI split
try:
    import langchain_community.llms as lc_community_llms
    from langchain_google_vertexai import ChatVertexAI, VertexAI

    # 1) Patch missing old module path:
    # ragas 내부에서 from langchain_community.chat_models.vertexai import ChatVertexAI 를 호출함
    vertexai_chat_module = types.ModuleType("langchain_community.chat_models.vertexai")
    vertexai_chat_module.ChatVertexAI = ChatVertexAI
    sys.modules["langchain_community.chat_models.vertexai"] = vertexai_chat_module

    # 2) Patch old attribute import:
    # ragas 내부에서 from langchain_community.llms import VertexAI 를 호출함
    setattr(lc_community_llms, "VertexAI", VertexAI)

    print("RAGAS VertexAI compatibility patch applied.")

except Exception as e:
    print("Patch failed:", type(e).__name__, e)

RAGAS VertexAI compatibility patch applied.


In [58]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model="gpt-4o-mini",
        temperature=1,
    )
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(
        model="text-embedding-3-small"
    )
)

print("RAGAS evaluator LLM and embeddings ready")

C:\Users\asguug\AppData\Local\Temp\ipykernel_16860\1093099365.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\asguug\AppData\Local\Temp\ipykernel_16860\1093099365.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision
C:\Users\asguug\AppData\Local\Temp\ipykernel_16860\1093099365.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections im

RAGAS evaluator LLM and embeddings ready


C:\Users\asguug\AppData\Local\Temp\ipykernel_16860\1093099365.py:16: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


In [59]:
ragas_metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
]

for metric in ragas_metrics:
    if hasattr(metric, "llm"):
        metric.llm = ragas_llm

    if hasattr(metric, "embeddings"):
        metric.embeddings = ragas_embeddings

print("RAGAS metrics configured")

RAGAS metrics configured


In [60]:
def run_ragas_for_retriever(ragas_input_df: pd.DataFrame, retriever_name: str) -> pd.DataFrame:
    """
    retriever 하나에 대해 RAGAS 평가를 수행한다.
    평가용 LLM/Embedding을 명시적으로 지정해 VertexAI 자동 import 문제를 피한다.
    """
    subset_df = ragas_input_df[ragas_input_df["retriever"] == retriever_name].copy()

    dataset = Dataset.from_pandas(
        subset_df[["question", "answer", "contexts", "ground_truth"]],
        preserve_index=False,
    )

    try:
        result = evaluate(
            dataset,
            metrics=ragas_metrics,
            llm=ragas_llm,
            embeddings=ragas_embeddings,
        )

    except TypeError:
        result = evaluate(
            dataset,
            metrics=ragas_metrics,
        )

    result_df = result.to_pandas()
    result_df["retriever"] = retriever_name

    return result_df

In [61]:
ragas_result_dfs = []

for retriever_name in ragas_input_df["retriever"].unique():
    print("RAGAS evaluating:", retriever_name)

    try:
        result_df = run_ragas_for_retriever(
            ragas_input_df=ragas_input_df,
            retriever_name=retriever_name,
        )
        ragas_result_dfs.append(result_df)

    except Exception as e:
        print(f"RAGAS failed for {retriever_name}")
        print(type(e).__name__, e)

if ragas_result_dfs:
    ragas_results_df = pd.concat(ragas_result_dfs, ignore_index=True)
    display(ragas_results_df.head())

    ragas_results_path = EVAL_DIR / "week5_ragas_results_by_question.csv"
    ragas_results_df.to_csv(ragas_results_path, index=False, encoding="utf-8-sig")

    print("saved:", ragas_results_path)

else:
    ragas_results_df = pd.DataFrame()
    print("RAGAS 결과가 생성되지 않았습니다.")

RAGAS evaluating: dense_only


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


RAGAS evaluating: bm25_only


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

...

RAGAS evaluating: hybrid_rerank


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,retriever
0,Hollow Knight는 어떤 플레이 스타일의 게임인가요?,[## Store Summary\nForge your own path in Holl...,Hollow Knight는 클래식한 2D 핸드드로우 스타일의 액션 어드벤처 게임입니...,"Hollow Knight는 2D 액션 어드벤처/메트로배니아 스타일의 게임으로, 탐험...",1.000,0.613411,0.95,dense_only
1,Hollow Knight의 분위기나 월드 구성은 어떤 특징이 있나요?,[Complete Hollow Knight to unlock Steel Soul M...,Hollow Knight의 분위기와 월드는 전반적으로 어둡고 서정적인 고딕풍 분위기...,Hollow Knight는 Hallownest라는 거대한 지하 세계를 배경으로 하며...,0.875,0.666492,1.00,dense_only
2,Monster Hunter: World의 핵심 플레이 루프는 무엇인가요?,[{STEAM_CLAN_IMAGE}/45725708/73392ac8e0f10e408...,간단히 말하면 반복되는 핵심 루프는 다음과 같습니다.\n\n- 퀘스트를 받고 다양한...,Monster Hunter: World의 핵심 플레이 루프는 몬스터를 추적하고 사냥...,0.750,0.422899,0.25,dense_only
3,Monster Hunter: World는 협동 플레이 측면에서 어떤 특징이 있나요?,[## About The Game\nWelcome to a new world! Ta...,"간단히 요약하면 다음과 같습니다.\n\n- 혼자 플레이 가능하며, 협동 플레이는 최...",Monster Hunter: World는 여러 플레이어가 함께 몬스터를 사냥하는 협...,1.000,0.468189,1.00,dense_only
4,Baldur's Gate 3는 어떤 RPG인가요?,[## Store Summary\nBaldur’s Gate 3 is a story-...,Baldur’s Gate 3는 던전 앤 드래곤즈 세계관을 바탕으로 한 스토리 중심의...,Baldur's Gate 3는 Dungeons & Dragons 규칙을 기반으로 한...,1.000,0.768762,1.00,dense_only


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_ragas_results_by_question.csv


In [62]:
if not ragas_results_df.empty:
    metric_cols = [
        col for col in ragas_results_df.columns
        if col in ["faithfulness", "answer_relevancy", "context_precision"]
    ]

    ragas_summary_df = (
        ragas_results_df
        .groupby("retriever", as_index=False)[metric_cols]
        .mean()
    )

    display(ragas_summary_df)

    ragas_summary_path = EVAL_DIR / "week5_ragas_summary.csv"
    ragas_summary_df.to_csv(ragas_summary_path, index=False, encoding="utf-8-sig")

    print("saved:", ragas_summary_path)

else:
    ragas_summary_df = pd.DataFrame()
    print("RAGAS summary를 만들 수 없습니다.")

,retriever,faithfulness,answer_relevancy,context_precision
0,bm25_only,0.974242,0.626140,0.478333
1,dense_only,0.887030,0.460830,0.683889
2,hybrid_rerank,0.933782,0.556704,0.828750
3,hybrid_rrf,0.927066,0.575172,0.838889


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_ragas_summary.csv


In [63]:
if not ragas_summary_df.empty:
    final_ablation_df = ragas_summary_df.merge(
        four_latency_summary_df[["retriever", "avg_latency_s"]],
        on="retriever",
        how="left",
    )

    final_ablation_df = final_ablation_df.sort_values(
        by=[col for col in ["context_precision", "answer_relevancy", "faithfulness"] if col in final_ablation_df.columns],
        ascending=False,
    )

    display(final_ablation_df)

    final_ablation_path = EVAL_DIR / "week5_final_ablation_summary.csv"
    final_ablation_df.to_csv(final_ablation_path, index=False, encoding="utf-8-sig")

    print("saved:", final_ablation_path)

else:
    final_ablation_df = four_latency_summary_df.copy()
    display(final_ablation_df)
    print("RAGAS 결과가 없어 latency summary만 표시합니다.")

,retriever,faithfulness,answer_relevancy,context_precision,avg_latency_s
3,hybrid_rrf,0.927066,0.575172,0.838889,0.029281
2,hybrid_rerank,0.933782,0.556704,0.828750,0.276223
1,dense_only,0.887030,0.460830,0.683889,0.031188
0,bm25_only,0.974242,0.626140,0.478333,0.000706


saved: C:\Users\asguug\Documents\rag-agent\data\eval\week5_final_ablation_summary.csv
